In [1]:
import pandas as pd
import plotly.express as px
import pycountry_convert as pc


In [2]:
POP_PATH = "/Users/tonytony/population.csv"

print("Population path:", POP_PATH)


Population path: /Users/tonytony/population.csv


In [3]:
pop_df = pd.read_csv(POP_PATH, skiprows=4)
print("Shape:", pop_df.shape)

pop_df.head()


Shape: (266, 70)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,108727.0,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,NaN
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,623369401.0,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,NaN
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,34700612.0,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,NaN
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,97630925.0,99706674.0,101854756.0,104089175.0,106388440.0,108772632.0,...,429454743.0,440882906.0,452195915.0,463365429.0,474569351.0,485920997.0,497387180.0,509398589.0,521764076.0,NaN
4,Angola,AGO,"Population, total",SP.POP.TOTL,5231654.0,5301583.0,5354310.0,5408320.0,5464187.0,5521981.0,...,29183070.0,30234839.0,31297155.0,32375632.0,33451132.0,34532429.0,35635029.0,36749906.0,37885849.0,NaN


In [4]:
pop_df = pop_df.drop(
    columns=[col for col in pop_df.columns if str(col).startswith("Unnamed")],
    errors="ignore"
)

year_cols = [col for col in pop_df.columns if str(col).isdigit()]
pop_df[year_cols] = pop_df[year_cols].apply(pd.to_numeric, errors="coerce")

print("Number of year columns:", len(year_cols))
print("Year range:", year_cols[0], "-", year_cols[-1])
print("Shape after cleaning:", pop_df.shape)

pop_df.head()


Number of year columns: 65
Year range: 1960 - 2024
Shape after cleaning: (266, 69)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,107906.0,108727.0,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,607123269.0,623369401.0,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,33831764.0,34700612.0,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,97630925.0,99706674.0,101854756.0,104089175.0,106388440.0,108772632.0,...,418127845.0,429454743.0,440882906.0,452195915.0,463365429.0,474569351.0,485920997.0,497387180.0,509398589.0,521764076.0
4,Angola,AGO,"Population, total",SP.POP.TOTL,5231654.0,5301583.0,5354310.0,5408320.0,5464187.0,5521981.0,...,28157798.0,29183070.0,30234839.0,31297155.0,32375632.0,33451132.0,34532429.0,35635029.0,36749906.0,37885849.0


In [5]:
SPECIAL_COUNTRY_CODES = {"XKX", "SXM"}

def is_country_code(code):
    code = str(code).strip()
    if code in SPECIAL_COUNTRY_CODES:
        return True
    try:
        alpha2 = pc.country_alpha3_to_country_alpha2(code)
        pc.country_alpha2_to_continent_code(alpha2)
        return True
    except:
        return False

pop_df["Is Country"] = pop_df["Country Code"].astype(str).apply(is_country_code)

print("Country / territory rows:", int(pop_df["Is Country"].sum()))
print("Aggregate rows:", int((~pop_df["Is Country"]).sum()))

pop_df[["Country Name", "Country Code", "Is Country"]].head()


Country / territory rows: 215
Aggregate rows: 51


,Country Name,Country Code,Is Country
0,Aruba,ABW,True
1,Africa Eastern and Southern,AFE,False
2,Afghanistan,AFG,True
3,Africa Western and Central,AFW,False
4,Angola,AGO,True


In [6]:
pop_geo = pop_df.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code", "Is Country"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Population, total"
)

pop_geo["Year"] = pd.to_numeric(pop_geo["Year"], errors="coerce").astype("Int64")
pop_geo["Population, total"] = pd.to_numeric(
    pop_geo["Population, total"],
    errors="coerce"
)

print("Long format shape:", pop_geo.shape)

pop_geo.head()


Long format shape: (17290, 7)


,Country Name,Country Code,Indicator Name,Indicator Code,Is Country,Year,"Population, total"
0,Aruba,ABW,"Population, total",SP.POP.TOTL,True,1960,54922.0
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,False,1960,130075728.0
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,True,1960,9035043.0
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,False,1960,97630925.0
4,Angola,AGO,"Population, total",SP.POP.TOTL,True,1960,5231654.0


In [7]:
pop_geo_clean = pop_geo[
    (pop_geo["Is Country"]) &
    (pop_geo["Population, total"].notna()) &
    (pop_geo["Population, total"] > 0)
].copy()

pop_geo_clean["Year Label"] = pop_geo_clean["Year"].astype(str)

print("Clean geo data shape:", pop_geo_clean.shape)
print("Year range:", int(pop_geo_clean["Year"].min()), "-", int(pop_geo_clean["Year"].max()))
print("Number of countries/territories:", pop_geo_clean["Country Code"].nunique())

pop_geo_clean.head()


Clean geo data shape: (13945, 8)
Year range: 1960 - 2024
Number of countries/territories: 215


,Country Name,Country Code,Indicator Name,Indicator Code,Is Country,Year,"Population, total",Year Label
0,Aruba,ABW,"Population, total",SP.POP.TOTL,True,1960,54922.0,1960
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,True,1960,9035043.0,1960
4,Angola,AGO,"Population, total",SP.POP.TOTL,True,1960,5231654.0,1960
5,Albania,ALB,"Population, total",SP.POP.TOTL,True,1960,1608800.0,1960
6,Andorra,AND,"Population, total",SP.POP.TOTL,True,1960,9510.0,1960


In [8]:
pop_coverage = (
    pop_geo_clean
    .groupby("Year")
    .agg(
        country_count=("Country Code", "nunique"),
        total_population=("Population, total", "sum"),
        min_population=("Population, total", "min"),
        median_population=("Population, total", "median"),
        mean_population=("Population, total", "mean"),
        max_population=("Population, total", "max")
    )
    .reset_index()
)

print("Population coverage by year:")
pop_coverage.tail(15)


Population coverage by year:


,Year,country_count,total_population,min_population,median_population,mean_population,max_population
50,2010,215,6.976885e+09,10043.0,5737971.0,3.245063e+07,1.337705e+09
51,2011,215,7.062670e+09,10098.0,5819051.0,3.284963e+07,1.345035e+09
52,2012,215,7.152372e+09,10267.0,5901287.0,3.326685e+07,1.354190e+09
53,2013,215,7.241228e+09,10517.0,5983845.0,3.368013e+07,1.363240e+09
54,2014,215,7.329331e+09,10742.0,6162955.0,3.408991e+07,1.371860e+09
55,2015,215,7.416847e+09,10954.0,6215770.0,3.449696e+07,1.379860e+09
56,2016,215,7.503979e+09,10930.0,6323060.0,3.490223e+07,1.387790e+09
57,2017,215,7.589555e+09,10869.0,6338660.0,3.530026e+07,1.396215e+09
58,2018,215,7.672219e+09,10751.0,6444079.0,3.568474e+07,1.402760e+09
59,2019,215,7.752951e+09,10581.0,6590211.0,3.606024e+07,1.407745e+09


In [12]:
year_order = [str(year) for year in sorted(pop_geo_clean["Year"].dropna().astype(int).unique())]

color_min = pop_geo_clean["Population, total"].quantile(0.02)
color_max = pop_geo_clean["Population, total"].quantile(0.98)

pop_animation = px.choropleth(
    pop_geo_clean,
    locations="Country Code",
    locationmode="ISO-3",
    color="Population, total",
    animation_frame="Year",
    animation_group="Country Code",
    hover_name="Country Name",
    hover_data={
        "Country Code": False,
        "Year": True,
        "Population, total": ":,.0f"
    },
    category_orders={"Year": year_order},
    color_continuous_scale="YlOrRd",
    range_color=[color_min, color_max],
    projection="natural earth",
    title="Population by Year",
    labels={
        "Population, total": "Population"
    }
)

pop_animation.update_layout(
    height=650,
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(title="Population")
)

pop_animation.show()


- Highly populated countries such as China, India, and the United States remain visually prominent across most years.
- Asia accounts for a very large share of the world population compared with other continents.
- Over time, many countries in Africa and Asia show clear population growth.
- Small countries and island nations have much lower populations, so they appear less prominent on the map.
- Population is strongly right-skewed because the gap between very large and very small countries is extremely large.

Conclusion: Population is suitable for analysis using totals by region or continent, as well as population shares. When visualizing population, the large gap between countries should be considered because a few highly populated countries can dominate the color scale.